In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/pseudo_labeled_tickets.csv")

print(df.shape)

(20000, 23)


In [5]:
embeddings = np.load("../data/embeddings.npy")

print(embeddings.shape)

(20000, 384)


In [6]:
mask = df["Severity_Delta"].isin(
    [-3, -2, 0, 2, 3]
)

df_clear = df[mask].copy()

embeddings_clear = embeddings[mask]

print(df_clear.shape)
print(embeddings_clear.shape)

print(df_clear["Mismatch_Label"].value_counts())

(11123, 23)
(11123, 384)
Mismatch_Label
0    5682
1    5441
Name: count, dtype: int64


In [7]:
meta_clear = df_clear[
    [
        "Issue_Category",
        "Ticket_Channel",
        "Resolution_Time_Hours"
    ]
]

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            [
                "Issue_Category",
                "Ticket_Channel"
            ]
        ),

        (
            "num",
            StandardScaler(),
            ["Resolution_Time_Hours"]
        )
    ]
)

meta_encoded = preprocessor.fit_transform(
    meta_clear
)

print(meta_encoded.shape)

(11123, 9)


In [9]:
X = np.hstack(
    [
        embeddings_clear,
        meta_encoded
    ]
)

y = df_clear["Mismatch_Label"]

print(X.shape)

(11123, 393)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(8898, 393)
(2225, 393)


In [22]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [24]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    recall_score
)

pred = model.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

print(
    classification_report(
        y_test,
        pred
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        pred,
        average="macro"
    )
)

print(
    "Recall Class 0:",
    recall_score(
        y_test,
        pred,
        pos_label=0
    )
)

print(
    "Recall Class 1:",
    recall_score(
        y_test,
        pred,
        pos_label=1
    )
)

Accuracy: 0.8013483146067416
              precision    recall  f1-score   support

           0       0.83      0.77      0.80      1137
           1       0.78      0.84      0.80      1088

    accuracy                           0.80      2225
   macro avg       0.80      0.80      0.80      2225
weighted avg       0.80      0.80      0.80      2225

Macro F1: 0.801293366114018
Recall Class 0: 0.7678100263852242
Recall Class 1: 0.8363970588235294


In [25]:
df_ultra = df[
    df["Severity_Delta"].isin([-3, 0, 3])
].copy()

print(df_ultra.shape)

print(df_ultra["Mismatch_Label"].value_counts())

print(df_ultra["Severity_Delta"].value_counts())

(6865, 23)
Mismatch_Label
0    5682
1    1183
Name: count, dtype: int64
Severity_Delta
 0    5682
 3     812
-3     371
Name: count, dtype: int64


In [26]:
mask = df["Severity_Delta"].isin([-3, 0, 3])

embeddings_ultra = embeddings[mask]

print(embeddings_ultra.shape)

(6865, 384)


In [27]:
meta_ultra = df_ultra[
    [
        "Issue_Category",
        "Ticket_Channel",
        "Resolution_Time_Hours"
    ]
]

In [28]:
meta_encoded = preprocessor.fit_transform(meta_ultra)

print(meta_encoded.shape)

(6865, 9)


In [29]:
X = np.hstack(
    [
        embeddings_ultra,
        meta_encoded
    ]
)

y = df_ultra["Mismatch_Label"]

print(X.shape)
print(y.value_counts())

(6865, 393)
Mismatch_Label
0    5682
1    1183
Name: count, dtype: int64


In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
scale_pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

print(scale_pos_weight)

4.805496828752642


In [32]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [33]:
pred = model.predict(X_test)

print(
    classification_report(
        y_test,
        pred
    )
)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

              precision    recall  f1-score   support

           0       0.97      0.95      0.96      1136
           1       0.79      0.87      0.83       237

    accuracy                           0.94      1373
   macro avg       0.88      0.91      0.89      1373
weighted avg       0.94      0.94      0.94      1373

Accuracy: 0.9373634377276038


In [34]:
import joblib

joblib.dump(
    model,
    "../models/final_xgb_model.pkl"
)

joblib.dump(
    preprocessor,
    "../models/final_preprocessor.pkl"
)

['../models/final_preprocessor.pkl']